# PublicNode (PERSISTENT DYNAMIC-EDGE)


## ⚙️ Stage 0: Global Runtime Config


In [ ]:
# --- PublicNode Core Config ---
import base64
import os
import secrets
import subprocess
import sys
import threading
import time
from pathlib import Path

os.environ['PYDEVD_DISABLE_FILE_VALIDATION'] = '1'
os.environ['PYTHONPATH'] = '/kaggle/working:/kaggle/working/vps-os'
VPS_NAME    = 'PublicNode'
VPS_VERSION = '0.1.0'

SIGNAL_TOPIC   = 'vps-root-ca623000b54f'
VPS_PASS_B64   = 'Tnp6LXlfdW5MUVZNU1lUZw=='
VPS_PASS       = base64.b64decode(VPS_PASS_B64).decode()
ENGINE_PORT    = 5003
GUI_PKGS       = 'thunar xfwm4 xfce4-panel xfce4-session xfdesktop4 xfce4-settings tumbler pm-utils upower xfce4-terminal x11-utils x11-xserver-utils dbus-x11 xfonts-base python3-xdg materia-gtk-theme papirus-icon-theme fonts-roboto mousepad'
BOOT_TIMEOUT   = 120
KAG_USER       = 'mohammadhasanulislam'
VAULT_SLUG     = 'vps-storage'
HF_REPO        = 'vps-vault'
HF_TOKEN       = ''
CREATOR        = 'Shesher Hasan'
ORGANIZATION   = 'myth-tools'
BUILD_TZ       = 'UTC'
GUI_ENABLED    = '{{GUI_ENABLED}}'
HEADLESS_MODE  = str(GUI_ENABLED).lower() != 'true'
GUI_RESOLUTION = '1920x1080'
GUI_DISPLAY    = ':1'
GUI_PORT       = 6080
BOOT_TIME      = time.time()

SESSION_ID     = secrets.token_hex(4)
os.environ['SESSION_ID'] = SESSION_ID
os.environ['VPS_PASS'] = VPS_PASS
os.environ['VPS_NAME'] = VPS_NAME
os.environ['VPS_VERSION'] = VPS_VERSION
os.environ['VPS_SIGNAL_TOPIC'] = SIGNAL_TOPIC
os.environ['KAG_USER'] = KAG_USER
os.environ['VAULT_SLUG'] = VAULT_SLUG
os.environ['HF_REPO'] = HF_REPO
os.environ['HF_TOKEN'] = HF_TOKEN
os.environ['CREATOR'] = CREATOR
os.environ['ORGANIZATION'] = ORGANIZATION
os.environ['GUI_ENABLED'] = GUI_ENABLED
os.environ['GUI_RESOLUTION'] = GUI_RESOLUTION
os.environ['GUI_DISPLAY'] = GUI_DISPLAY
os.environ['GUI_PORT'] = str(GUI_PORT)
os.environ['ORGANIZATION'] = ORGANIZATION

def _acquire_singleton_lock():
    pid_file = '/tmp/vps_master.pid'
    if os.path.exists(pid_file):
        try:
            with open(pid_file) as f:
                old_pid = int(f.read().strip())
            os.kill(old_pid, 0)
            print(f'\n[SYSTEM] Master process already active (PID: {old_pid}). Exiting silently.\n', flush=True)
            os._exit(0)
        except (OSError, ValueError):
            os.remove(pid_file)
    with open(pid_file, 'w') as f:
        f.write(str(os.getpid()))

_acquire_singleton_lock()
sys.dont_write_bytecode = True
os.system('mkdir -p logs && truncate -s 0 logs/master.log')
# --- Pre-flight: purge any stale processes and MOTD noise ---
os.system('pkill -9 cloudflared || true')
os.system('pkill -9 sshd || true')
os.system('pkill -9 tmux || true')
os.system('rm -rf /tmp/tmux* /tmp/vps_tmux.sock || true')
# Industrial MOTD & Ubuntu Detail Purge (V7.0)
os.system('touch /root/.hushlogin')

# --- Build Shell Profile (V5.3: Robust tmux Backbone) ---
Path('/root/.bashrc').write_text("""
export PS1='-(root@PublicNode)-[\\w]\\n-\\${debian_chroot:+(\\${debian_chroot})} \\$ '
alias l='lsd -l --group-directories-first'
alias ll='lsd -la --group-directories-first'

# Auto-attach to persistent tmux session on login (inside PRoot)
if [[ -z "$TMUX" && "$TERM" != "screen" ]]; then
    exec /usr/local/bin/proot -r /kaggle/working/proot_root -b /kaggle/working -b /proc -b /dev -b /sys -w /root tmux -S /tmp/vps_tmux.sock -u new -A -s vps
fi
""")


## ⚔️ Stage 1: Kernel Tuning


In [ ]:
import os
import time
class VpsArmor:
    @staticmethod
    def log(m, s='◢◤'):
        msg = f'{s} {m}'
        print(f'\n{msg}\n', flush=True)
        try:
            os.makedirs('logs', exist_ok=True)
            with open('logs/master.log', 'a') as f:
                f.write(f'[{time.strftime("%H:%M:%S")}] {msg}\n')
        except Exception:
            pass
        try:
            os.system(f'curl -s -d "STATUS: [{SESSION_ID}] {msg}" https://ntfy.sh/{SIGNAL_TOPIC}')
        except Exception:
            pass
    @staticmethod
    def broadcast(m):
        """Send a base64-encoded signal for WS/PASS as expected by the mobile app."""
        b64 = base64.b64encode(m.encode()).decode()
        try:
            os.system(f'curl -s -d "{b64}" https://ntfy.sh/{SIGNAL_TOPIC}')
        except Exception:
            pass
    @staticmethod
    def wait_locks():
        VpsArmor.log('WAITING FOR SYSTEM PACKAGE LOCKS...', '⏳')
        for _ in range(30):  # Wait max 60s
            if os.system('pgrep -x "apt|apt-get|dpkg"') != 0:
                return
            time.sleep(2)
        VpsArmor.log('FORCING LOCK RELEASE (TIMED OUT)...', '⚔️')
        os.system('pkill -9 apt || true')
        os.system('pkill -9 apt-get || true')
        os.system('pkill -9 dpkg || true')
        os.system('rm -f /var/lib/apt/lists/lock /var/cache/apt/archives/lock /var/lib/dpkg/lock* || true')
        os.system('dpkg --configure -a || true')
    @staticmethod
    def robust_install():
        if os.path.exists('/tmp/.vps_provisioning_active'):
            VpsArmor.log('PROVISIONING ALREADY IN PROGRESS. SKIPPING REDUNDANT CALL.', '⏳')
            return
        os.system('touch /tmp/.vps_provisioning_active')
        VpsArmor.log('SYNCING PUBLICNODE ASSETS...')
        VpsArmor.wait_locks()
        # Purge broken or unreachable sources that often hang Kaggle boots
        os.system('rm -f /etc/apt/sources.list.d/*r2u* || true')
        # V9.2: Aggressively nuke any source mentioning launchpad if it's causing timeouts
        os.system('grep -l "launchpad" /etc/apt/sources.list.d/*.list | xargs rm -f || true')
        os.system('sed -i "s/^deb-src/# deb-src/" /etc/apt/sources.list /etc/apt/sources.list.d/* || true')
        
        # V11.0: Industrial-Grade Mirror Optimization
        VpsArmor.log('OPTIMIZING APT MIRRORS (mirrors.kernel.org)...', '🚀')
        os.system('sed -i "s|http://archive.ubuntu.com/ubuntu/|http://mirrors.kernel.org/ubuntu/|g" /etc/apt/sources.list || true')
        os.system('sed -i "s|http://security.ubuntu.com/ubuntu/|http://mirrors.kernel.org/ubuntu/|g" /etc/apt/sources.list || true')
        
        # V11.1: Persistent APT Cache Redirection
        VpsArmor.log('CONFIGURING PERSISTENT APT CACHE...', '📦')
        os.system('mkdir -p /kaggle/working/.cache/apt/archives/partial && rm -rf /var/cache/apt/archives && ln -s /kaggle/working/.cache/apt/archives /var/cache/apt/archives')
        
        # APT/UV Resilience: Prevent redundant work across kernel restarts
        if os.path.exists('/tmp/.provisioning_done'):
            VpsArmor.log('PROVISIONING ALREADY COMPLETE. SKIPPING.', '✅')
            return
        # V10.4: ET Provisioning moved to background for speed.
        # V10.5: Optimized Sprint-Mode Provisioning (Parallel Wget, Zero Lock Contention)
        VpsArmor.log('STARTING BACKGROUND PROVISIONING (uv, cloudflared, websocat, proot)...', '⚡')
        os.system('mkdir -p /usr/local/bin /root/.cache/huggingface || true')
        os.system('''
            (
                # 1. UV Engine
                [ ! -f /usr/local/bin/uv ] && wget -q https://github.com/astral-sh/uv/releases/latest/download/uv-x86_64-unknown-linux-musl.tar.gz -O uv.tar.gz && \
                tar -xzf uv.tar.gz && mv uv-x86_64-unknown-linux-musl/uv /usr/local/bin/uv && rm -rf uv.tar.gz uv-x86_64-unknown-linux-musl &
                
                # 2. Cloudflared
                [ ! -f /usr/local/bin/cloudflared ] && wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared &
                
                # 3. Websocat
                [ ! -f /usr/local/bin/websocat ] && wget -q https://github.com/vi/websocat/releases/latest/download/websocat.x86_64-unknown-linux-musl -O /usr/local/bin/websocat && chmod +x /usr/local/bin/websocat &
                
                # 4. PRoot Engine
                [ ! -f /usr/local/bin/proot ] && wget -q https://proot.gitlab.io/proot/bin/proot -O /usr/local/bin/proot && chmod +x /usr/local/bin/proot &
                
                wait
                touch /tmp/.provisioning_done
            ) &''')
        # Eternal Terminal: High-Resilience Multi-Mirror Download
        if not os.path.exists('/usr/bin/et'):
             VpsArmor.log('PREPARING ETERNAL TERMINAL...', '📡')
             distro = os.popen('lsb_release -cs').read().strip()
             if distro == 'jammy':
                 mirrors = [
                     "https://raw.githubusercontent.com/myth-tools/PublicNode/main/Packages/et_6.2.10-jammy1_amd64.deb",
                     "https://launchpad.net/~jgmath2000/+archive/ubuntu/et/+files/et_6.2.10-jammy1_amd64.deb",
                     "https://ppa.launchpadcontent.net/jgmath2000/et/ubuntu/pool/main/e/et/et_6.2.10-jammy1_amd64.deb"
                 ]
             else:
                 mirrors = [
                     "https://launchpad.net/~jgmath2000/+archive/ubuntu/et/+files/et_6.2.8-focal2_amd64.deb"
                 ]
             for url in mirrors:
                 VpsArmor.log(f'TRYING ET MIRROR: {url[:50]}...', '📡')
                 # Use curl with 20s timeout and auto-redirect following
                 if os.system(f'curl -L -s -m 20 -f {url} -o /tmp/et.deb') == 0:
                     if os.path.exists('/tmp/et.deb') and os.path.getsize('/tmp/et.deb') > 102400:
                         VpsArmor.log(f'ET DOWNLOAD SUCCESSFUL: {url[:50]}...', '✅')
                         break
             
             if os.path.exists('/tmp/et.deb') and os.path.getsize('/tmp/et.deb') > 102400:
                 VpsArmor.log('INSTALLING ETERNAL TERMINAL...', '📡')
                 os.system('DEBIAN_FRONTEND=noninteractive apt-get install -y /tmp/et.deb || true')
             else:
                 VpsArmor.log('ET INSTALLATION SKIPPED: ALL MIRRORS TIMED OUT OR FAILED.', '⚠️')
             os.system('rm -f /tmp/et.deb || true')
             os.system('pkill -9 -f Xvfb; pkill -9 -f x11vnc; true')
        
        # APT Resilience: Set timeouts and aggressive retries for connection issues
        apt_opts = '-o Acquire::Retries=3 -o Acquire::http::Timeout="30" -o Acquire::https::Timeout="30"'
        
        # V9.3: Run update ONCE before the loop to save time (Kaggle apt mirrors are slow)
        os.system(f'apt-get update {apt_opts}')
        
        # V10.5: Single Atomic Install (Conditional GUI)
        VpsArmor.log('INSTALLING SYSTEM PACKAGES...')
        extra_pkgs = f'{GUI_PKGS if not HEADLESS_MODE else ""}'
        install_res = os.system(f'DEBIAN_FRONTEND=noninteractive apt-get install -y --no-install-recommends --fix-missing {apt_opts} curl wget git sudo zsh zstd tar fzf bat fd-find htop jq xz-utils git-lfs openssh-server {extra_pkgs}')
        VpsArmor.log(f'PACKAGE INSTALLATION FINISHED (Exit Code: {install_res})', '📦')
        os.system('rm -f /tmp/et.deb || true')
        
        # Move MOTD/SSH Purge HERE (After openssh-server is installed)
        os.system('rm -f /etc/update-motd.d/* /etc/motd /etc/legal /var/run/motd.dynamic || true')
        os.system('truncate -s 0 /etc/motd /etc/legal /var/run/motd.dynamic || true')
        os.system('sed -i "s/PrintMotd yes/PrintMotd no/" /etc/ssh/sshd_config || true')
        os.system('sed -i "s/PrintLastLog yes/PrintLastLog no/" /etc/ssh/sshd_config || true')
        os.system('sed -i "/Banner/d" /etc/ssh/sshd_config || true')
        
        # Final Stabilization: Wait for background tasks (uv, cloudflared)
        VpsArmor.log('STABILIZING ENGINES...', '⏳')
        for _ in range(60):
            if os.path.exists('/tmp/.provisioning_done'): break
            time.sleep(1)
        if not os.path.exists('/tmp/.vps_py_ready'):
            VpsArmor.log('FINALIZING PYTHON ENVIRONMENT...', '🐍')
            os.system('/usr/local/bin/uv pip install --system --upgrade --no-cache kaggle huggingface_hub psutil requests fastapi uvicorn[standard] python-multipart watchdog hf_transfer')
            os.system('touch /tmp/.vps_py_ready')
        
        # --- PublicNode System Vault Pre-flight ---
        os.makedirs('/root/.cache/huggingface', exist_ok=True)
        # --- PublicNode Persistence Prep ---
        os.system('mkdir -p /root/.ssh && chmod 700 /root/.ssh')
        if HF_TOKEN:
            with open('/root/.cache/huggingface/token', 'w') as f:
                f.write(HF_TOKEN)
        VpsArmor.log('SYSTEM VAULT: Ready for boot-time restoration.', '🔐')
        if not os.path.exists('/kaggle/working/proot_root'):
            os.makedirs('/kaggle/working/proot_root', exist_ok=True)
            # SMART RECOVERY: The Engine will pull the latest snapshot from HuggingFace on boot.
            # Here we just initialize a fresh base if nothing exists yet.
            recovered = False
            if not recovered:
                VpsArmor.log('SYSTEM VAULT: Initializing ephemeral environment...', '📦')
        # Fail-safe checks for critical system binaries
        if not os.path.exists('/usr/bin/zsh'):
            os.system(f'apt-get install -y --no-install-recommends {apt_opts} zsh')
        os.system('chsh -s /usr/bin/zsh root || true')
        if not os.path.exists('/usr/sbin/sshd'):
            os.system(f'apt-get install -y --no-install-recommends {apt_opts} openssh-server')
        # Robust Oh-My-Zsh installation
        if not os.path.exists('/root/.oh-my-zsh'):
            os.system('sh -c "$(curl -fsSL https://raw.githubusercontent.com/ohmyzsh/ohmyzsh/master/tools/install.sh)" "" --unattended')
        VpsArmor.log('POLISHING SHELL ARCHITECTURE...', '🐚')
        os.system(f'hostname {VPS_NAME} 2>/dev/null || true')
        os.system('mkdir -p /root/.oh-my-zsh/custom/plugins')
        os.system('git clone -q https://github.com/zsh-users/zsh-autosuggestions /root/.oh-my-zsh/custom/plugins/zsh-autosuggestions')
        os.system('git clone -q https://github.com/zsh-users/zsh-syntax-highlighting /root/.oh-my-zsh/custom/plugins/zsh-syntax-highlighting')
        
        # Premium PublicNode Zsh Configuration
        bashrc_parts = [
            'export ZSH="/root/.oh-my-zsh"',
            'ZSH_THEME=""',
            'plugins=(git zsh-autosuggestions zsh-syntax-highlighting fzf)',
            'source $ZSH/oh-my-zsh.sh',
            'export TERM=xterm-256color',
            '',
            '# PublicNode Premium Prompt',
            'NEWLINE=$' + "'" + '\\n' + "'" + '',
            "PROMPT='%F{81}┌──(%B%F{33}%n%b%f%F{81}⬢%B%F{33}PublicNode%b%f%F{81})─[%B%F{white}%~%b%F{81}]${NEWLINE}└─%B%F{33}$%b%f '",
            '',
            '# Modern CLI Aliases (with fallback checks)',
            '[ -x "$(command -v lsd)" ] && alias ls="lsd" || alias ls="ls --color=auto"',
            '[ -x "$(command -v lsd)" ] && alias l="lsd -l" || alias l="ls -l"',
            '[ -x "$(command -v lsd)" ] && alias la="lsd -a" || alias la="ls -A"',
            '[ -x "$(command -v lsd)" ] && alias ll="lsd -la" || alias ll="ls -la"',
            '[ -x "$(command -v batcat)" ] && alias cat="batcat --paging=never"',
            '[ -x "$(command -v fdfind)" ] && alias find="fdfind"',
            'alias top="htop"',
            '',
            '# PublicNode Industrial Telemetry Banner (V8.1)',
            'OS_NAME=$(grep ^NAME= /etc/os-release | cut -d "=" -f 2 | tr -d \'"\')',
            'OS_VER=$(grep ^VERSION_ID= /etc/os-release | cut -d "=" -f 2 | tr -d \'"\')',
            'KERNEL=$(uname -r)',
            'UPTIME=$(uptime -p | sed "s/up //")',
            'CPU_MODEL=$(grep "model name" /proc/cpuinfo | head -n 1 | cut -d ":" -f 2 | xargs || uname -m)',
            'CPU_CORES=$(nproc)',
            'MEM_TOTAL=$(free -h | grep Mem | awk \'{print $2}\')',
            'STR_WORK=$(df -h /kaggle/working | tail -1 | awk \'{print $3 " / " $2}\')',
            'STR_ROOT=$(df -h / | tail -1 | awk \'{print $2}\')',
            '# V9: Ultra-Accurate Sync Status (Aware of active background saves)',
            'if [ -f /kaggle/working/logs/vps_save_history.log ]; then STR_INF="SYNCED"; ',
            'elif [ -f /tmp/vps_sync_lock ]; then STR_INF="SYNCING..."; ',
            'else STR_INF="UNSYNCED"; fi',
            'echo -e "\033[1;36m◣◥◤◢  \033[1;37mPUBLICNODE OS \033[1;32mONLINE \033[1;36m ◣◥◤◢\033[0m"',
            'echo -e "\033[1;36mDISTRIBUTION: \033[0m$OS_NAME $OS_VER ($KERNEL)"',
            'echo -e "\033[1;36mUPTIME:       \033[0m$UPTIME"',
            'echo -e "\033[1;36mPROCESSOR:    \033[0m$CPU_MODEL ($CPU_CORES Cores)"',
            'echo -e "\033[1;36mMEMORY:       \033[0m$MEM_TOTAL"',
            'echo -e "\033[1;36mSTORAGE:      \033[0mWORKSPACE: $STR_WORK • SYSTEM: $STR_ROOT"',
            'echo -e "\033[1;36mSYSTEM VAULT: \033[1;32m$STR_INF\033[0m"',
            'echo -e "\033[1;36mCREATOR:      \033[0mShesher Hasan"',
            'echo -e "\033[1;36mORGANIZATION: \033[0mmyth-tools"',
            '# --- Master PublicNode GUI & Environment Bridge ---',
            'export XDG_RUNTIME_DIR=/tmp/runtime-root',
            'mkdir -p $XDG_RUNTIME_DIR && chmod 700 $XDG_RUNTIME_DIR',
            'if [ -S /tmp/.X11-unix/X1 ]; then',
            '    export DISPLAY=:1',
            '    [ -f /tmp/vps_dbus_env ] && . /tmp/vps_dbus_env > /dev/null 2>&1',
            '    [ -f /tmp/vps_electron_env ] && . /tmp/vps_electron_env > /dev/null 2>&1',
            '    # Self-Healing: Verify D-Bus process is actually alive',
            '    if ! ps -p "$DBUS_SESSION_BUS_PID" > /dev/null 2>&1; then eval $(dbus-launch --sh-syntax); fi',
            '    export NO_AT_BRIDGE=1',
            'fi',
            '# --- Universal Forensic Aliases ---',
            'alias sudo="sudo -E"',
            '# --- End Master Bridge ---',
            'echo ""'
        ]
        bashrc_content = chr(10).join(bashrc_parts)
        for f_path in ['/root/.bashrc', '/root/.zshrc']:
            try:
                with open(f_path, 'a') as f:
                    f.write(chr(10) + bashrc_content + chr(10))
            except Exception: pass
        VpsArmor.log('RESOURCES ARMORED')
os.chdir('/kaggle/working')
VpsArmor.log(f'WORKSPACE AUDIT: {os.listdir(".")}', '🔍')
for d in ['logs', 'vps-os']:
    Path(d).mkdir(exist_ok=True, parents=True)
VpsArmor.log('MATERIALIZING ASSETS...', '')
OS_ENGINE_B64 = '...'
with open('vps-os/vps_os_engine.py', 'wb') as f:
    f.write(base64.b64decode(OS_ENGINE_B64))
VpsArmor.log('OS CORE MATERIALIZED')
# --- Injection: Universal 'vps' CLI for remote use ---
tpl_vps = r'''#!/bin/bash
C_CYAN="\033[1;36m"
C_NC="\033[0m"
C_RED="\033[1;31m"
C_YELLOW="\033[1;33m"
C_GREEN="\033[1;32m"
case "$1" in
  sync)   
       echo -e "${C_CYAN}[VAULT]${C_NC} Triggering System Vault Sync..."
       res=$(curl -s localhost:__PORT__/api/sync)
       if [[ $(echo "$res" | jq -r ".status") != "accepted" ]]; then
           echo -e "${C_RED}[ERROR]${C_NC} Sync rejected: $(echo "$res" | jq -r ".message")" ; exit 1
       fi
       echo -e "${C_YELLOW}[SYNC]${C_NC} Job system-vault-sync initiated. Polling status..."
       while true; do
           job=$(curl -s localhost:__PORT__/api/sync/status)
           active=$(echo "$job" | jq -r ".active")
           phase=$(echo "$job" | jq -r ".phase")
           progress=$(echo "$job" | jq -r ".progress")
           msg=$(echo "$job" | jq -r ".message")
           ver=$(echo "$job" | jq -r ".version")
           err_msg=$(echo "$job" | jq -r ".error")
           if [[ "$active" == "false" ]]; then
               if [[ "$err_msg" != "null" && -n "$err_msg" ]]; then
                   echo -e "\n${C_RED}[CRITICAL]${C_NC} Sync Failed: $err_msg" ; exit 1
               fi
               echo -e "\n${C_GREEN}[SUCCESS]${C_NC} System Vault Sync Complete (Version: $ver). Backbone locked." ; break
           fi
           printf "\r${C_CYAN}[%3d%%]${C_NC} Phase: %-10s | %-45s" "$progress" "$phase" "$msg" ; sleep 2
       done ;;
  vault)
    case "$2" in
      push)  echo -e "${C_CYAN}[VAULT]${C_NC} Pushing $3 to Private Vault..."; curl -s "localhost:__PORT__/api/vault/push?path=$3" ;;
      list)  echo -e "${C_CYAN}[VAULT]${C_NC} Listing Private Vault contents:"; curl -s "localhost:__PORT__/api/vault/list" | jq -r '.files[]' ;;
      *)     echo -e "Usage: vps vault [push <path> | list]" ;;
    esac ;;
  top)    echo -e "${C_CYAN}[PULSE]${C_NC} Recent Processes:"; curl -s localhost:__PORT__/api/system/pulse | grep -oP '\{"pid":\d+,"name":"[^"]+","cpu_percent":[\d.]+' | head -n 12 | sed 's/\{"pid"://; s/,"name":"/ | /; s/", "cpu_percent":/ | CPU: /; s/$/% /' ;;
  logs)   echo -e "${C_CYAN}[SYSTEM]${C_NC} Streaming Telemetry..."; tail -f /kaggle/working/logs/master.log ;;
  status) curl -s localhost:__PORT__/api/stats ;;
  *)      echo -e "◢◤ PublicNode Remote CLI v__VER__\nUsage: vps [sync|vault|top|logs|status]" ;;
esac
'''
remote_cli = tpl_vps.replace('__PORT__', '5003').replace('__VER__', '0.1.0')
with open("/usr/local/bin/vps", "w") as f:
    f.write(remote_cli)
os.system("chmod +x /usr/local/bin/vps")
# --- Injection: Autonomous Interceptors (apt/pip/npm) ---
SYSTEM_STATE_DIR = '/kaggle/working/vault/system_state'
os.system(f'mkdir -p {SYSTEM_STATE_DIR}')
tpl_wrapper = r'''#!/bin/bash
CMD=$(basename "$0")
REAL_CMD=""
for p in /usr/bin /bin /usr/sbin /sbin; do
  if [[ -x "$p/$CMD" && "$p" != "/usr/local/bin" ]]; then REAL_CMD="$p/$CMD"; break; fi
done
if [[ -z "$REAL_CMD" ]]; then REAL_CMD=$(which -a "$CMD" | grep -v "/usr/local/bin" | head -n 1); fi
if [[ -z "$REAL_CMD" ]]; then echo "Command not found"; exit 1; fi
"$REAL_CMD" "$@"
RET=$?
if [ $RET -eq 0 ]; then
  if [[ " $* " =~ " install " || " $* " =~ " upgrade " || " $* " =~ " remove " ]]; then
    echo "$CMD $@" >> __STATE_DIR__/restore.sh
  fi
fi
exit $RET
'''
wrapper = tpl_wrapper.replace('__STATE_DIR__', SYSTEM_STATE_DIR)
for cmd in ['apt', 'apt-get', 'pip', 'npm']:
    with open(f'/usr/local/bin/{cmd}', 'w') as f:
        f.write(wrapper)
os.system('chmod +x /usr/local/bin/apt /usr/local/bin/apt-get /usr/local/bin/pip /usr/local/bin/npm')
if os.path.exists('.vps_auth/kaggle.json'):
    os.system('mkdir -p ~/.kaggle && cp .vps_auth/kaggle.json ~/.kaggle/kaggle.json && chmod 600 ~/.kaggle/kaggle.json')
try:
    VpsArmor.robust_install()
except Exception as e:
    VpsArmor.log(f'CRITICAL BOOT FAILURE: {e}', '❌')
    import traceback
    traceback.print_exc()
    sys.exit(1)


## 🔄 Stage 2: PublicNode Restoration


In [ ]:
if os.path.exists('/kaggle/input/vps-storage'):
    # Kaggle mounts datasets as uncompressed directories automatically
    # We just need to sync the contents back to /kaggle/working
    VpsArmor.log('RESTORE PULSE: Synchronizing State from Vault', '📦')
    ret = os.system('rsync -a /kaggle/input/vps-storage/ /kaggle/working/')
    if ret == 0:
        VpsArmor.log('RESTORE COMPLETE')
    else:
        VpsArmor.log(f'RESTORE FAILED (exit {ret}). Continuing with clean workspace.', '⚠️')
    if os.path.exists('/kaggle/working/vault/system_state/restore.sh'):
        VpsArmor.log('RESTORING SYSTEM STATE...', '⚙️')
        os.system('bash /kaggle/working/vault/system_state/restore.sh > logs/restore_state.log 2>&1 &')

    # User Vault Sync from HuggingFace is now handled by the VPS OS Engine sentinel
    # to ensure background auto-revival and atomic updates.


## 🛰️ Stage 3: Launch Engines


In [ ]:
VpsArmor.log('LAUNCHING ENGINE...')
# Ensure no stale processes conflict
os.system('pkill -9 uvicorn || true; pkill -9 cloudflared || true; true')
def start_os(): os.system(f'GUI_ENABLED={GUI_ENABLED} PYDEVD_DISABLE_FILE_VALIDATION=1 VPS_VERSION=0.1.0 VPS_ENGINE_PORT={ENGINE_PORT} VPS_PASS={VPS_PASS} SESSION_ID={SESSION_ID} VPS_SIGNAL_TOPIC={SIGNAL_TOPIC} VPS_CONTROL_TOPIC={SIGNAL_TOPIC}-control HF_TOKEN={HF_TOKEN} HF_REPO={HF_REPO} python3 -Xfrozen_modules=off -u vps-os/vps_os_engine.py')
if not os.path.exists('/tmp/.vps_engine_ignited'):
    os.system('touch /tmp/.vps_engine_ignited')
    threading.Thread(target=start_os, daemon=True).start()
else:
    VpsArmor.log('ENGINE ALREADY IGNITED. ATTACHING TO EXISTING INSTANCE.', '🔗')

def start_sshd():
    if True:
        VpsArmor.log('ARMORING SSH ARCHITECTURE...', '🔑')
        os.system('mkdir -p /root/.ssh && echo "ssh-ed25519 AAAAC3NzaC1lZDI1NTE5AAAAIPtfN5KBIfk0z+BSFGUt2GuLPhC06OgZRCuGm1K4Pmj8 shesher@shesher" >> /root/.ssh/authorized_keys && chmod 700 /root/.ssh && chmod 600 /root/.ssh/authorized_keys')
        os.system(f'echo "root:{VPS_PASS}" | chpasswd')
        os.system('mkdir -p /run/sshd')
        os.system('rm -f /etc/motd /etc/update-motd.d/*')
        os.system('sed -i "s/^[# ]*PermitRootLogin.*/PermitRootLogin yes/" /etc/ssh/sshd_config')
        os.system('sed -i "s/^[# ]*PasswordAuthentication.*/PasswordAuthentication yes/" /etc/ssh/sshd_config')
        os.system('sed -i "s/^#UseDNS.*/UseDNS no/" /etc/ssh/sshd_config || echo "UseDNS no" >> /etc/ssh/sshd_config')
        os.system('sed -i "s/^PrintMotd.*/PrintMotd no/" /etc/ssh/sshd_config || echo "PrintMotd no" >> /etc/ssh/sshd_config')
        os.system('sed -i "s/^PrintLastLog.*/PrintLastLog no/" /etc/ssh/sshd_config || echo "PrintLastLog no" >> /etc/ssh/sshd_config')
        os.system('sed -i "s/^#ClientAliveInterval.*/ClientAliveInterval 300/" /etc/ssh/sshd_config || echo "ClientAliveInterval 300" >> /etc/ssh/sshd_config')
        os.system('sed -i "s/^#TCPKeepAlive.*/TCPKeepAlive yes/" /etc/ssh/sshd_config || echo "TCPKeepAlive yes" >> /etc/ssh/sshd_config')
        os.system('sed -i "s/^#IPQoS.*/IPQoS lowdelay throughput/" /etc/ssh/sshd_config || echo "IPQoS lowdelay throughput" >> /etc/ssh/sshd_config')
        os.system('/usr/sbin/sshd -p 22')
        if os.path.exists('/usr/bin/et'):
            os.system('mkdir -p logs')
            os.system('etserver --daemon --port 2022 --log_dir logs > logs/et_init.log 2>&1')
        # WebSocket-to-SSH Bridge for mobile app clients (localhost only)
        if os.path.exists('/usr/local/bin/websocat'):
            os.system('/usr/local/bin/websocat -E -b ws-l:127.0.0.1:40008 tcp:127.0.0.1:22 &')
            VpsArmor.log('WEBSOCKET BRIDGE ARMED (127.0.0.1:40008)', '📱')

threading.Thread(target=start_sshd, daemon=True).start()


## 📊 Stage 4: PublicNode Backbone


In [ ]:
import re
import socket
import time

import requests

# Inherit global config
BOOT_TIMEOUT = 120
BOOT_TIME    = int(time.time())
for _ in range(BOOT_TIMEOUT):
    try:
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
            s.settimeout(1)
            if s.connect_ex(('127.0.0.1', 22)) == 0:
                VpsArmor.log('SSH DAEMON READY. IGNITING BACKBONE...', '🚀')
                break
    except Exception:
        pass
    time.sleep(1)
else:
    VpsArmor.log('SSH FAILED TO RESPOND. BACKBONE MAY FAIL.', '⚠️')

VpsArmor.log('LAUNCHING TRIPLE BACKBONE (SSH + ET + WS)...', '🚀')
# Start tunnels
for f in ['logs/cf_ssh.log', 'logs/cf_ws.log', 'logs/cf_et.log', 'logs/cf_api.log', 'logs/cf_gui.log']: 
    if os.path.exists(f):
        os.remove(f)

if True:
    os.system('/usr/local/bin/cloudflared tunnel --no-autoupdate --url tcp://127.0.0.1:22 > logs/cf_ssh.log 2>&1 &')
    if os.path.exists('/usr/bin/et'):
        os.system('/usr/local/bin/cloudflared tunnel --no-autoupdate --url tcp://127.0.0.1:2022 > logs/cf_et.log 2>&1 &')
    if os.path.exists('/usr/local/bin/websocat'):
        os.system('/usr/local/bin/cloudflared tunnel --no-autoupdate --url http://127.0.0.1:40008 > logs/cf_ws.log 2>&1 &')
    if GUI_ENABLED == 'true':
        os.system(f'/usr/local/bin/cloudflared tunnel --no-autoupdate --url http://127.0.0.1:6080 > logs/cf_gui.log 2>&1 &')
    os.system('/usr/local/bin/cloudflared tunnel --no-autoupdate --url http://127.0.0.1:5003 > logs/cf_api.log 2>&1 &')

def get_cf_url(log_file):
    if os.path.exists(log_file):
        with open(log_file) as f:
            content = f.read()
            # V9.9: Broad-Spectrum Regex for Cloudflare URL Detection
            match = re.search(r'((?:https|tcp)://[a-zA-Z0-9.-]+\.(?:trycloudflare\.com|cloudflareaccess\.com|direct))', content)
            if match:
                return match.group(1)
    return None

ssh_url = None
et_url = None
ws_url = None
api_url = None
gui_url = None

VpsArmor.log('WAITING FOR BACKBONE LOCKS...', '⏳')
for i in range(15):
    if not ssh_url:
        ssh_url = get_cf_url('logs/cf_ssh.log')
    if not et_url:
        et_url = get_cf_url('logs/cf_et.log')
    if not ws_url:
        ws_url = get_cf_url('logs/cf_ws.log')
    if not api_url:
        api_url = get_cf_url('logs/cf_api.log')
    if GUI_ENABLED == 'true' and not gui_url:
        gui_url = get_cf_url('logs/cf_gui.log')
    if ws_url:
        ws_url = ws_url.replace('http://', 'wss://').replace('https://', 'wss://')
    if i % 3 == 0:
        VpsArmor.log(f'SCANNING CHANNELS (ATTEMPT {i+1}/45)...', '📡')
    if (ssh_url and ws_url and api_url and (not os.path.exists('/usr/bin/et') or et_url)):
        break
    time.sleep(2)

if not ssh_url:
    VpsArmor.log('SSH BACKBONE LOCK FAILED. DUMPING DIAGNOSTICS:', '❌')
    os.system('tail -n 20 logs/cf_ssh.log')
    raise RuntimeError('SSH BACKBONE LOCK FAILED')
if not api_url:
    VpsArmor.log('API BACKBONE LOCK FAILED. DUMPING DIAGNOSTICS:', '❌')
    os.system('tail -n 20 logs/cf_api.log')
    raise RuntimeError('API BACKBONE LOCK FAILED')

def check_backbone(url):
    try:
        # Use curl to check for 'online' status in JSON response
        cmd = ['curl', '-s', '-L', '--max-time', '5', url]
        res = subprocess.run(cmd, capture_output=True, text=True, timeout=7, check=False)
        return 'online' in res.stdout.lower()
    except Exception:
        return False

VpsArmor.log('STABILIZING BACKBONE...', '⚖️')
for _ in range(20):
    # Ensure the API engine is fully responsive before unlocking the backbone
    if check_backbone(api_url):
        time.sleep(3) # Final stabilization buffer
        break
    time.sleep(2)

VpsArmor.log(f'SSH BRIDGE LOCKED -> {ssh_url}', '')
VpsArmor.broadcast(f'SSH:[{SESSION_ID}]{ssh_url}')
if et_url:
    VpsArmor.log(f'ECHO BRIDGE LOCKED -> {et_url}', '')
    VpsArmor.broadcast(f'ET:[{SESSION_ID}]{et_url}')
if ws_url:
    VpsArmor.log(f'APP BRIDGE LOCKED -> {ws_url}', '📱')
    VpsArmor.broadcast(f'WS:[{SESSION_ID}]{ws_url}')
if api_url:
    VpsArmor.log(f'API BRIDGE LOCKED -> {api_url}', '⚡')
    VpsArmor.broadcast(f'API:[{SESSION_ID}]{api_url}')
if gui_url:
    VpsArmor.log(f'GUI BRIDGE LOCKED -> {gui_url}', '🖥️')
    VpsArmor.broadcast(f'GUI:[{SESSION_ID}]{gui_url}')
VpsArmor.broadcast(f'PASS:[{SESSION_ID}]{VPS_PASS}')

VpsArmor.log('PUBLICNODE PERSISTENCE ONLINE', '✅')
VpsArmor.log(f'SSH URL: {ssh_url}', '')
if et_url:
    import re
    et_host = re.sub(r'^.*?://', '', et_url)
    ssh_host = re.sub(r'^.*?://', '', ssh_url)
    VpsArmor.log(f'ET COMMAND: et --ssh-option="Hostname={ssh_host}" --ssh-option="Port=443" root@{et_host}:443', '🐚')
def notebook_sentinel():
    while True:
        try:
            r = requests.get(f'https://ntfy.sh/{SIGNAL_TOPIC}-control/raw?poll=1', timeout=10)
            for line in r.text.splitlines():
                if 'KILL:' in line:
                    try:
                        t = int(line.split(':')[1])
                        if t > BOOT_TIME:
                            VpsArmor.log('PUBLICNODE KILL SIGNAL RECEIVED. SECURING VAULT...', '🛑')
                            try:
                                requests.get('http://localhost:5003/api/system/save', timeout=120)
                                # Poll for completion before exiting
                                for _ in range(60):
                                    res = requests.get('http://localhost:5003/api/sync/status', timeout=10)
                                    state = res.json()
                                    if not state.get('active'):
                                        break
                                    time.sleep(5)
                            except Exception as sync_err:
                                VpsArmor.log(f'VAULT SECURE FAILED: {sync_err}', '❌')
                            VpsArmor.log('VAULT SECURED. GOODBYE.', '⚰️')
                            os._exit(0)
                    except (IndexError, ValueError):
                        pass
                elif 'SAVE:' in line:
                    try:
                        VpsArmor.log('PUBLICNODE SAVE SIGNAL RECEIVED. SECURING VAULT...', '💾')
                        requests.get('http://localhost:5003/api/system/save', timeout=120)
                    except Exception:
                        pass
        except Exception:
            pass
        time.sleep(15)

threading.Thread(target=notebook_sentinel, daemon=True).start()
try:
    while True:
        time.sleep(60)
except KeyboardInterrupt:
    VpsArmor.log('KEYBOARD INTERRUPT — Shutting down gracefully.')
